In [5]:
import os
import time
import json
import hashlib
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta

base_dir = "rtm1_2501010372"
os.makedirs(f"{base_dir}/notebooks", exist_ok=True)
os.makedirs(f"{base_dir}/data/raw", exist_ok=True)
os.makedirs(f"{base_dir}/docs", exist_ok=True)

WIKI_API_URL = "https://id.wikipedia.org/w/api.php"
headers = {
    'User-Agent': 'InstikiInformaticsStudent-YulianaFarida-2501010372 (2501010372@instiki.ac.id)'
}

categories = [
    "Kategori:Pariwisata_di_Bali",
    "Kategori:Budaya_Bali",
    "Kategori:Pura_di_Bali",
    "Kategori:Tarian_Bali",
    "Kategori:Masakan_Bali"
]
all_items = []

print("Mengirim request ke Wikipedia Indonesia API...")
for cat in categories:
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": cat,
        "cmlimit": 60,
        "cmtype": "page",
        "format": "json"
    }
    response = requests.get(WIKI_API_URL, params=params, headers=headers, timeout=30)
    if response.status_code == 200:
        members = response.json().get("query", {}).get("categorymembers", [])
        for m in members:
            all_items.append({
                "page_id": m.get("pageid"),
                "title": m.get("title"),
                "category": cat.replace("Kategori:", "").replace("_", " "),
                "namespace": m.get("ns"),
                "source_url": f"https://id.wikipedia.org/?curid={m.get('pageid')}"
            })
        print(f"Berhasil mengambil {len(members)} data dari {cat}")
    else:
        print(f"Gagal mengambil {cat}. Status code: {response.status_code}")

    time.sleep(1.5)

df = pd.DataFrame(all_items).drop_duplicates(subset=['page_id']).head(130)

wita = timezone(timedelta(hours=8))
now = datetime.now(wita)
date_str = now.strftime('%Y%m%d')

csv_filename = f"wikipedia_bali_budaya_{date_str}.csv"
csv_path = f"{base_dir}/data/raw/{csv_filename}"
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"\nDataset mentah tersimpan: {csv_path} ({len(df)} baris, {len(df.columns)} kolom)")

file_size = os.path.getsize(csv_path)
with open(csv_path, "rb") as f:
    md5_hash = hashlib.md5(f.read()).hexdigest()

provenance_data = {
    "file_name": csv_filename,
    "file_size_bytes": file_size,
    "md5_hash": md5_hash,
    "source_name": "Wikipedia Bahasa Indonesia (MediaWiki Action API)",
    "source_url": WIKI_API_URL,
    "license": "CC BY-SA 4.0 - Wikimedia Foundation",
    "acquisition_timestamp": now.isoformat(),
    "collector_name": "Yuliana Farida",
    "collector_nim": "2501010372"
}

with open(f"{base_dir}/docs/provenance.json", "w", encoding='utf-8') as f:
    json.dump(provenance_data, f, indent=4)

ethics_rows = [
    [1, "Legalitas Sumber Data", "Terpenuhi", "Menggunakan MediaWiki Action API resmi berlisensi terbuka CC BY-SA 4.0."],
    [2, "Pemeriksaan Ketentuan / robots.txt", "Terpenuhi", "Mematuhi Wikimedia API Etiquette dan batas rate limit resmi."],
    [3, "Atribusi Sumber", "Terpenuhi", "Mencantumkan atribusi Wikipedia Bahasa Indonesia & Wikimedia Foundation."],
    [4, "User-Agent Jujur", "Terpenuhi", "Menyertakan nama lengkap Yuliana Farida dan email kampus 2501010372@instiki.ac.id."],
    [5, "Jeda Antar-Request", "Terpenuhi", "Menerapkan jeda waktu 1.5 detik antar kategori menggunakan time.sleep(1.5)."],
    [6, "Penanganan Timeout & Status Code", "Terpenuhi", "Mengatur timeout 30 detik dan pengecekan response.status_code == 200."],
    [7, "Tanpa Data Pribadi", "Terpenuhi", "Hanya mengekstrak metadata artikel publik sektor pariwisata & budaya Bali."],
    [8, "Keamanan Kredensial", "Terpenuhi", "Tidak menggunakan API key maupun kredensial rahasia di dalam kode notebook."],
    [9, "Keutuhan Data Mentah", "Terpenuhi", "File di data/raw/ disimpan langsung dari hasil ekstraksi tanpa modifikasi manual."]
]
pd.DataFrame(ethics_rows, columns=["no", "butir_etika", "status", "bukti_implementasi"]).to_csv(
    f"{base_dir}/docs/data_ethics_checklist.csv", index=False, encoding='utf-8'
)

readme_content = f"""# Tugas 1: Akuisisi Data Nyata (BIFP-243)
- **Nama:** Yuliana Farida
- **NIM:** 2501010372
- **Kelas:** 3 / F — Program Studi Informatika, INSTIKI

## 1. Sumber Data
- **Nama Sumber:** Wikipedia Bahasa Indonesia (MediaWiki Action API)
- **URL:** {WIKI_API_URL}
- **Penerbit:** Wikimedia Foundation
- **Alasan Pemilihan:** Menyediakan daftar terstruktur objek pariwisata, warisan budaya, pura, seni tari, dan kuliner di Bali.

## 2. Tanggal dan Jam Akses
- **Waktu Akses:** {now.strftime('%d %B %Y, pukul %H:%M:%S WITA')}

## 3. Metode Akuisisi
- **Teknik:** Pengambilan data melalui MediaWiki Action API menggunakan pustaka `requests` Python.
- **Endpoint:** `{WIKI_API_URL}`
- **Parameter Query:** `action=query`, `list=categorymembers` untuk kategori `Pariwisata_di_Bali`, `Budaya_Bali`, `Pura_di_Bali`, `Tarian_Bali`, dan `Masakan_Bali`.

## 4. Batasan Etika
- **Ketentuan & Lisensi:** Berlisensi CC BY-SA 4.0 dan mematuhi Wikimedia API Etiquette.
- **Jeda & Rate Limit:** Diberikan jeda `time.sleep(1.5)` antar-request dan batas `timeout=30` detik.
- **Data yang Tidak Diambil:** Tidak mengambil data personal individu maupun informasi sensitif apa pun.

## 5. Penjelasan Kolom Dataset
1. `page_id`: Nomor identifikasi unik halaman pada basis data Wikipedia.
2. `title`: Judul artikel objek wisata, budaya, pura, tarian, atau kuliner Bali.
3. `category`: Kategori tematik data (`Pariwisata di Bali`, `Budaya Bali`, `Pura di Bali`, `Tarian Bali`, `Masakan Bali`).
4. `namespace`: Kode ruang nama halaman Wikipedia (`0` untuk artikel utama).
5. `source_url`: Tautan permanen menuju halaman artikel sumber.

## 6. Cara Menjalankan Ulang Notebook
1. Pastikan Python 3 beserta pustaka `requests` dan `pandas` sudah terinstal (`pip install requests pandas`).
2. Buka berkas `notebooks/akuisisi_wikipedia.ipynb`.
3. Pilih menu **Run All** untuk menjalankan seluruh sel dari awal hingga akhir.
4. Berkas dataset mentah dan dokumentasi akan otomatis diperbarui di folder masing-masing.
"""

with open(f"{base_dir}/README.md", "w", encoding='utf-8') as f:
    f.write(readme_content)

print("Selesai! Tabel data di bawah ini:")
df.head(10)

Mengirim request ke Wikipedia Indonesia API...
Berhasil mengambil 1 data dari Kategori:Pariwisata_di_Bali
Berhasil mengambil 60 data dari Kategori:Budaya_Bali
Berhasil mengambil 38 data dari Kategori:Pura_di_Bali
Berhasil mengambil 1 data dari Kategori:Tarian_Bali
Berhasil mengambil 0 data dari Kategori:Masakan_Bali

Dataset mentah tersimpan: rtm1_2501010372/data/raw/wikipedia_bali_budaya_20261116.csv (100 baris, 5 kolom)
Selesai! Tabel data di bawah ini:


,page_id,title,category,namespace,source_url
0,2161950,Pariwisata di Bali,Pariwisata di Bali,0,https://id.wikipedia.org/?curid=2161950
1,4640836,Aci Sanghyang Grodog,Budaya Bali,0,https://id.wikipedia.org/?curid=4640836
2,4645486,Aci Tulak Punggul,Budaya Bali,0,https://id.wikipedia.org/?curid=4645486
3,277626,Aksara Bali,Budaya Bali,0,https://id.wikipedia.org/?curid=277626
4,143082,Arja,Budaya Bali,0,https://id.wikipedia.org/?curid=143082
5,3949539,Arja geguntangan,Budaya Bali,0,https://id.wikipedia.org/?curid=3949539
6,4985,Astawara,Budaya Bali,0,https://id.wikipedia.org/?curid=4985
7,74300,Ayam betutu,Budaya Bali,0,https://id.wikipedia.org/?curid=74300
8,4549034,Baris Upacara,Budaya Bali,0,https://id.wikipedia.org/?curid=4549034
9,1012131,Barong Bali,Budaya Bali,0,https://id.wikipedia.org/?curid=1012131
